# 13. Asking a language model directly (zero-shot, GPU)

The baseline for "just ask the model": Mistral-Small-24B **Instruct** (the instruct version of the frozen model of step
7, 4 bit) reads each anonymous article with a description of NewsGuard's nine criteria and answers with the score
(0–100) of the outlet that published it. Nothing is trained; greedy decoding; the outlet score is the mean over its
articles, as for every model (prompt and parsing in `zeroshot.py`). Outputs in `predictions/regression/LLM-zero-shot/`,
so steps 8, 9 and 15 include it like the models of step 7.

In [1]:
LIMIT = 0                     # > 0: only the first LIMIT articles, to test the prompt (nothing is saved)
GPU_GB = 20

In [2]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())

import models as MO
import zeroshot as ZS

art, outlets = MO.load()
print(ZS.SYSTEM, "\n\n" + ZS.USER.format(text="<article>"))

{'python': '3.12.3', 'seed': 42, 'split seed': 1, 'numpy': '2.5.4', 'pandas': '3.0.6', 'sklearn': '1.9.1', 'scipy': '1.18.1', 'torch': '2.5.1+cu121', 'transformers': '4.51.3', 'bitsandbytes': '0.50.2', 'trafilatura': '2.3.1'}


You are an expert media analyst. NewsGuard rates news and information websites with a score from 0 to 100, the weighted sum of nine journalistic criteria: the site does not repeatedly publish false content (22 points), gathers and presents information responsibly (18), regularly corrects or clarifies errors (12.5), handles the difference between news and opinion responsibly (12.5), avoids deceptive headlines (10), discloses ownership and financing (7.5), clearly labels advertising (7.5), reveals who is in charge, including possible conflicts of interest (5), and provides the names of content creators with contact or biographical information (5). Scores below 60 mean that the site generally fails to meet basic standards of credibility and transparency. 

Below is one article published by a news website. The names of news outlets have been replaced by [OUTLET], the names of journalists by [AUTHOR], links by [URL] and e-mail addresses by [EMAIL].

Estimate the NewsGuard score (0-100) of t

/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [3]:
if LIMIT or not ZS.ANSWERS.exists():                   # the GPU only to ask the model
    MO.wait_for_gpu(GPU_GB, LOG)
res = ZS.run(art, LOG, limit=LIMIT or None)
print(f"{len(res):,} answers; without a number (given the mean training score): {(~res.number).sum()}")
res.answer.value_counts().head(15)

2026-11-16 14:00:35  zero-shot: answers already saved


2026-11-16 14:00:35  zero-shot: done, 24 answers without a number (given the mean training score)


25,524 answers; without a number (given the mean training score): 24


answer
65                                15134
45                                 6532
40                                  985
60                                  574
75                                  453
35                                  407
55                                  329
25                                  323
30                                  285
85                                  219
20                                  106
22                                   90
Based on the provided article,       24
67                                   19
68                                   17
Name: count, dtype: int64

In [4]:
res.prediction.describe().round(1).to_frame("predicted score per article").T

,count,mean,std,min,25%,50%,75%,max
predicted score per article,25524.0,57.3,11.8,0.0,45.0,65.0,65.0,95.0
